# Classification de déchets avec PySpark MLlib

Ce notebook montre comment entraîner un modèle de Machine Learning pour identifier le type de déchet à partir de vos images préalablement traitées par `data_tansformation.py`.

In [6]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, array_position
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, DoubleType
from datetime import datetime

spark = SparkSession.builder \
    .appName("GarbageClassificationML") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark


### Chargement et formatage des données
Keras nécessite que les features soient sous forme de `Veteurs`. Nos pixels sont stockés au format texte dans un .parquet

In [5]:
def load_and_prepare_data(path, split):
    df = spark.read.parquet(path)
    label_col = f"y_{split}"
    feature_col = f"x_{split}"

    df = df.withColumn(
        "label",
        array_position(col(label_col), 1) - 1
    )

    return df.select("label", col(feature_col).alias("features")).dropna()

# Test
train_data = load_and_prepare_data("../data/train_data.parquet", "train")
test_data = load_and_prepare_data("../data/test_data.parquet", "test")
train_data.show(5)
test_data.show(5)


PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

In [ ]:
CONFIG = {
    "img_size": 64,
    "batch_size": 64,
    "epochs": 60,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "seed": 42
}

def data_augmentation():
    data_aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.05)
    ], name="data_augmentation")
    return data_aug


def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def spark_to_numpy(df):

    pdf = df.toPandas()

    
    y = pdf["label"].to_numpy(dtype=np.int32)

    
    X = np.array(pdf["features"].tolist(), dtype=np.float32)

    X = X.reshape(X.shape + (1,))

    return X, y

def save_model(model, name):
    filename = f"{name}.keras"
    save_path = os.path.join("..", "models", filename)
    model.save(save_path)
    print(f"Modèle sauvegardé : {save_path}")

def create_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        data_augmentation(),
        tf.keras.layers.Rescaling(1.0 / 255.0),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ], name="CNN")
    
    return compile_model(model)


def main():
    tf.keras.utils.set_random_seed(CONFIG["seed"])

    train_prepared = train_data.cache()

    train_split_df, val_df = train_prepared.randomSplit([0.9, 0.1], seed=CONFIG["seed"])

    X_train, y_train = spark_to_numpy(train_split_df)
    X_val, y_val = spark_to_numpy(val_df)

    input_shape = X_train.shape[1:]
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print(f"Distribution des images sur les labels: {np.bincount(y_val)}")

    num_classes = int(max(y_train.max(), y_val.max()) + 1)

    class_counts = np.bincount(y_train, minlength=num_classes)
    class_weight = {
        i: float(len(y_train) / (num_classes * count))
        for i, count in enumerate(class_counts) if count > 0
    }

    model = create_cnn(input_shape, num_classes)

    model.summary()

    #Entraînement
    log_name = f"{model.name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    current_log_path = os.path.join("..", "models", "logs", log_name)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"
            ),
            tf.keras.callbacks.TensorBoard(log_dir=current_log_path)
        ],
        verbose=2
    )
    save_model(model, f"final_{model.name}")
    spark.catalog.clearCache()


main()


Train: (6602, 64, 64, 1), Val: (722, 64, 64, 1)
Distribution des images sur les labels: [157 101 192  83 112  77]


Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,126,502 (8.11 MB)

 Trainable params: 2,126,310 (8.11 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/60
